# 07.6 — Walkthrough: the Soft Three-Stage Curriculum

This is the capstone of Module 07: all three levers ([07.3](07.3_load_parameters.ipynb) load, [07.4](07.4_loss_weights_curriculum.ipynb) weights, [07.5](07.5_freeze_unfreeze_curriculum.ipynb) freeze) traced *together*, from the actual shipped preset, epoch by epoch. We'll load `"Soft Three-Stage Curriculum - Shortened"` — the Optimal regime — with one function call, advance it through training, and watch the three schedules choreograph a coherent story: **train the autoencoder on clean data, then bring in the classifier as noise ramps up, then freeze the representation and fine-tune the classifier back on clean data.** One `CurriculumBundle`, one `update(epoch)` per epoch.

This notebook covers:

* The `CurriculumBundle` — three schedules + KL anneal behind one `update()`.
* Loading the real preset by name and tracing every lever.
* The bundle's update order and how the training loop consumes it.
* Reading the whole curriculum as three coordinated stages.

**Prerequisites:** [07.3](07.3_load_parameters.ipynb), [07.4](07.4_loss_weights_curriculum.ipynb), [07.5](07.5_freeze_unfreeze_curriculum.ipynb).

## Section 1 — What MATLAB does

`cgg_trainNetwork` builds the three dynamic-parameter objects once, then re-evaluates them every epoch inside the loop:

```matlab
LoadParameters = cgg_generateLoadParameters_v2(regime);
LossWeights    = cgg_generateLossWeights_v2(regime);
FreezeParams   = cgg_generateFreezeParameters(regime);

for epoch = 1:numEpochs
    loads   = cgg_calculateDynamicValue(LoadParameters, epoch);   % augmentation stds
    weights = cgg_calculateDynamicValue(LossWeights,    epoch);   % loss weights
    frozen  = cgg_calculateDynamicValue(FreezeParams,   epoch);   % LR factors
    net = setLearnRateFactor(net, frozen);
    % ... train one epoch with loads (data), weights (loss), frozen (LR) ...
end
```

`regime = "Soft Three-Stage Curriculum - Shortened"` selects the Optimal schedule. The port collapses the three `cgg_generate*` objects into a single `CurriculumBundle` and the per-epoch re-evaluation into `bundle.update(epoch)`.

## Section 2 — The Python concepts you need

### 2.1 — The `CurriculumBundle`: three schedules, one handle

`load_curriculum_by_name` reads the preset YAML and builds a `CurriculumBundle` — the load, weight, and freeze `Schedule`s plus the optional `KLBaseAnneal` ([07.4 §2.4](07.4_loss_weights_curriculum.ipynb)) — so the loop advances the entire curriculum with a single call:

In [ ]:
from neural_data_decoding.training.schedules.library import load_curriculum_by_name

bundle = load_curriculum_by_name(
    "Soft Three-Stage Curriculum - Shortened",
    base_loads={"std_white_noise": 1.0, "std_channel_offset": 1.0, "std_random_walk": 1.0},
    base_weights={"reconstruction": 1.0, "kl": 1.0, "classification": 1.0},
    base_freezes={"encoder": 1.0, "decoder": 1.0, "classifier": 1.0},
)
print("load lever  :", bundle.load.names())
print("weight lever:", bundle.weight.names())
print("freeze lever:", bundle.freeze.names())
print("→ one bundle, three Schedules — the whole Optimal curriculum from a preset name.")

### 2.2 — Trace every lever, epoch by epoch

In [ ]:
print("epoch | noise | recon | class | enc-frz | cls-frz | stage")
for e in (1, 5, 10, 15, 20, 25, 30, 45):
    bundle.update(e)
    noise = bundle.load.current("std_white_noise")
    recon = bundle.weight.current("reconstruction")
    clas  = bundle.weight.current("classification")
    encf  = bundle.freeze.current("encoder")
    clsf  = bundle.freeze.current("classifier")
    stage = "reconstruct" if clas < 0.5 else ("handoff" if recon > 0.5 else "classify")
    print(f"  {e:2}  | {noise:.2f}  | {recon:.2f}  | {clas:.2f}  |  {encf:.3f}  |  {clsf:.3f}  | {stage}")

Read across the rows and the three levers move as one:

* **Epochs 1–10:** noise ≈ 0.01 (clean), reconstruction full, classification ≈ 0.01 *and* the classifier is frozen (0.01). The autoencoder learns to reconstruct clean data; the classifier is inert.
* **Epochs 10–20:** noise ramps to full, classification weight *and* the classifier's LR factor ramp up together. The classifier starts learning, on a now-useful latent, as the data gets harder.
* **Epochs 20–30:** reconstruction weight decays *and* the encoder freezes (0.208 → 0.01). The representation is locked; the classifier (full weight, unfrozen) owns the objective.
* **Epoch 45:** noise decays back toward clean (0.059), reconstruction ≈ 0.01, encoder frozen, classifier full. Fine-tune the classifier on near-clean data with a frozen encoder.

Notice how the weight lever and the freeze lever *agree* at every step — classification's weight rises exactly as the classifier unfreezes; reconstruction's weight decays exactly as the encoder freezes. That coordination is the curriculum's design ([07.5 §2.5](07.5_freeze_unfreeze_curriculum.ipynb)).

### 2.3 — The curriculum dashboard

In [ ]:
import matplotlib.pyplot as plt

epochs = list(range(1, 50))
noise, recon, clas, encf, clsf = [], [], [], [], []
for e in epochs:
    bundle.update(e)
    noise.append(bundle.load.current("std_white_noise"))
    recon.append(bundle.weight.current("reconstruction"))
    clas.append(bundle.weight.current("classification"))
    encf.append(bundle.freeze.current("encoder"))
    clsf.append(bundle.freeze.current("classifier"))

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
axes[0].plot(epochs, noise, color="#7a4a1a", label="white-noise std"); axes[0].set_ylabel("LOAD\n(augmentation)"); axes[0].legend(loc="upper right")
axes[1].plot(epochs, recon, color="#4a7a1a", label="reconstruction"); axes[1].plot(epochs, clas, color="crimson", label="classification"); axes[1].set_ylabel("WEIGHTS\n(loss)"); axes[1].legend(loc="center right")
axes[2].plot(epochs, encf, color="#1a4a7a", label="encoder"); axes[2].plot(epochs, clsf, color="crimson", label="classifier"); axes[2].set_ylabel("FREEZE\n(LR factor)"); axes[2].set_yscale("log"); axes[2].legend(loc="center right")
for ax in axes:
    ax.axvspan(1, 10, alpha=0.07, color="green"); ax.axvspan(10, 20, alpha=0.07, color="orange"); ax.axvspan(20, 49, alpha=0.07, color="red")
axes[0].set_title("The Soft Three-Stage Curriculum — three levers, one timeline")
axes[2].set_xlabel("epoch")
plt.tight_layout(); plt.show()
print("green=reconstruct, orange=handoff, red=classify. All three levers tell the same story.")

### 2.4 — The update order (and the KL base-anneal)

`bundle.update(epoch)` applies its pieces in a specific order, mirroring `cgg_trainNetwork`: **first** rewrite the KL weight's base from the `KLBaseAnneal` ramp ([07.4 §2.4](07.4_loss_weights_curriculum.ipynb)), **then** update load, weight, and freeze. The KL base rewrite must come first because the weight schedule's dynamic multiply is applied *on top of* the annealed base — order matters for getting the effective KL weight right.

In [ ]:
from neural_data_decoding.training.schedules.bundle import KLBaseAnneal

# Attach a KL base-anneal and watch the KL weight get the two-layer treatment:
bundle.kl_anneal = KLBaseAnneal(initial_weight=1.0, delay_epoch=5, epoch_ramp=10)
for e in (3, 10, 25):
    bundle.update(e)
    print(f"epoch {e:2}: KL weight base (post-anneal) = {bundle.weight['kl'].base:.3f}, "
          f"current (× dynamic) = {bundle.weight.current('kl'):.3f}")
print("→ update() rewrites the KL base from the anneal FIRST, then the dynamic multiply scales it.")

### 2.5 — How the training loop consumes the bundle

Each epoch, `fit_supervised` does exactly four curriculum-related things, in order:

1. `curriculum.update(epoch + 1)` — advance all levers (the `+1` converts Python's 0-indexed epoch to MATLAB's 1-indexed, [07.2 §3](07.2_piecewise_linear_schedules.ipynb)).
2. `apply_freeze_to_optimizer(optimizer, curriculum.freeze, base_lr=...)` — write the freeze LR factors ([07.5](07.5_freeze_unfreeze_curriculum.ipynb)).
3. `_resolve_epoch_loss_weights(loss_weights, curriculum)` — snapshot the weight lever into a dict ([07.4 §2.5](07.4_loss_weights_curriculum.ipynb)).
4. Train the epoch — the *load* lever is read *live* inside the dataset's `__getitem__` ([07.3 §2.4](07.3_load_parameters.ipynb)), not here.

So the three levers reach the model through three different channels at three different cadences: freeze → optimizer at epoch start; weights → a per-epoch snapshot; load → live per-batch. One bundle, three delivery mechanisms — but a single `update(epoch)` drives them all.

## Section 3 — The `neural_data_decoding` implementation

`CurriculumBundle.update` — the KL-base-first ordering:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/schedules/bundle.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "def update" in line and "epoch" in line)
end = next((j for j in range(i + 1, len(src)) if src[j].strip().startswith("def ") or src[j].startswith("__all__")), len(src))
for k in range(i, min(end, i + 16)):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* If `kl_anneal` is set and `"kl"` is in the weight schedule, `update` **first** rewrites `weight["kl"].base = kl_anneal.value_at(epoch)` — the base-anneal before the dynamic multiply (§2.4).
* Then `load.update(epoch)`, `weight.update(epoch)`, `freeze.update(epoch)` — all three levers advanced by the one call.
* The preset comes from `configs/schedule/soft_three_stage_curriculum_shortened.yaml` via `load_curriculum_by_name`; `slugify_regime` turns the MATLAB regime name into the filename (`"Soft Three-Stage Curriculum - Shortened"` → `soft_three_stage_curriculum_shortened`).
* The whole subsystem is parity-tested against MATLAB golden fixtures at `1e-12` (`tests/parity/test_t2_dynamic_schedule_*_parity.py`) — including the full Soft Three-Stage regime per-epoch and the off-by-one quirk ([07.2](07.2_piecewise_linear_schedules.ipynb)).

## Section 4 — Hands-on exercises

### Exercise 4.1 — the three stages are self-consistent

For each stage, verify the "active" subnetwork's loss weight and freeze factor *agree* (both high in its stage, both low otherwise).

In [ ]:
# Reveal:
for e, label in [(5, "reconstruct"), (20, "handoff"), (40, "classify")]:
    bundle.kl_anneal = None            # isolate the schedule levers
    bundle.update(e)
    print(f"epoch {e:2} ({label:11}): "
          f"class weight={bundle.weight.current('classification'):.2f}, "
          f"classifier freeze={bundle.freeze.current('classifier'):.2f}  "
          f"→ {'agree (both high)' if bundle.weight.current('classification') > 0.5 and bundle.freeze.current('classifier') > 0.5 else 'classifier not yet active'}")
print("→ the classifier's weight and its trainability rise together — coordinated levers.")

### Exercise 4.2 — swap in a simpler regime

Load the `"KL Annealing"` preset (which drives *only* the KL weight) and confirm its load and freeze levers are static while KL ramps.

In [ ]:
# Reveal:
kl_only = load_curriculum_by_name("KL Annealing",
    base_weights={"reconstruction": 1.0, "kl": 1.0, "classification": 1.0})
for e in (1, 50, 100):
    kl_only.update(e)
    print(f"epoch {e:3}: kl weight={kl_only.weight.current('kl'):.4f}, "
          f"classification weight={kl_only.weight.current('classification'):.2f} (static)")
print("→ 'KL Annealing' ramps only KL (1e-4 → 1.0 over epochs 10–100); other levers stay put.")

## Section 5 — Common errors

### The curriculum has no effect

`bundle.update(epoch)` must be called each epoch, and each lever consumed through its channel (§2.5): freeze → `apply_freeze_to_optimizer`, weights → `_resolve_epoch_loss_weights`, load → the dataset's live read. Miss any wiring and that lever goes silent.

### KL weight wrong because of update order

The KL base-anneal must be applied *before* the dynamic multiply (§2.4). `CurriculumBundle.update` does this; a hand-rolled loop that multiplies first, then rewrites the base, gets the wrong effective KL weight.

### Wrong preset name

`load_curriculum_by_name` slugifies the regime name to a filename. A typo → `FileNotFoundError`. The four shipped presets: `None`, `No Dynamic Parameters`, `KL Annealing`, `Soft Three-Stage Curriculum - Shortened`.

### Levers disagree (weight up, subnetwork frozen)

In a well-designed regime they're coordinated (§2.2). If you author a custom regime, keep the weight and freeze schedules consistent — training a subnetwork whose loss is down-weighted (or vice versa) wastes the curriculum's intent.

### Off-by-one surprises at stage boundaries

Every lever inherits the [07.2](07.2_piecewise_linear_schedules.ipynb) off-by-one — values reach their waypoint magnitude one epoch late. At a stage boundary, expect the transition to complete one epoch after the nominal waypoint. That's parity-correct, not a bug.

## Section 6 — Further reading

- [`src/neural_data_decoding/training/schedules/bundle.py`](../../src/neural_data_decoding/training/schedules/bundle.py) — `CurriculumBundle`, `KLBaseAnneal`.
- [`src/neural_data_decoding/training/schedules/library.py`](../../src/neural_data_decoding/training/schedules/library.py) — `load_curriculum_by_name`, `slugify_regime`.
- [`configs/schedule/soft_three_stage_curriculum_shortened.yaml`](../../configs/schedule/soft_three_stage_curriculum_shortened.yaml) — the Optimal preset itself.

**Module 07 complete.** You've traced the dynamic curriculum end to end: the intuition, the piecewise interpolator (and its quirk), and the three levers — load augmentation, loss weights (with KL annealing), and freeze factors — culminating in the full Soft Three-Stage regime driven by one `CurriculumBundle`. Next up: **[Module 08 — output & analysis](../08_output_and_analysis/)**, where a trained model's predictions become the confusion-matrix tables and metrics the pipeline reports.